# Dense Retrieval

Dense Retrieval is relies on neural embeddings. Both the query and the documents are converted into embeddings. This provides a continuous vector space. 

## Load in documents and chunk them

The documents are chunked so the vector store does not receive a massive amount of documents at once. the chunk size is the length of 

In [1]:
from rag.documents import load_documents
from rag.chunk import chunk_documents

FILE_PATH = "/home/nick/github-projects/Sec-Rag/data/google_10K.pdf"

documents = load_documents(FILE_PATH)

chunks = chunk_documents(
    documents=documents,
    chunk_size=400,
    chunk_overlap=40
)

print(f"Length of Documents: {len(documents)}")
print(f"Length of Chunks: {len(chunks)}")

/home/nick/github-projects/Sec-Rag/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Length of Documents: 107
Length of Chunks: 1037


## Initialize VectorStore and ingest embeddings and chunks into vector store for retrieval.

The vector store needs to have both the chunks and the embedding model so the chunked documents are converted into embeddings during retrieval.

In [2]:
from rag.dense import DenseRetriever

retriever = DenseRetriever()
retriever.add_documents(chunks)

print(f"Indexed {len(retriever)} chunks")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5164.35it/s]


Indexed 1037 chunks


## Retrieve for a query

The query is embedded with the same model as the chunks and the closest vectors come back.

In [3]:
query = "how much did Google spend on research and development in 2025"

results = retriever.retrieve(query, top_k=5)

for doc in results:
    print(f"\n[page {doc.metadata['page']}]\n{doc.page_content[:200]}")


[page 41]
2025, primarily due to a revenue mix shift from Google Network properties to Google Search & other properties. The
TAC rates on Google Search & other and Google Network revenues were substantially con

[page 40]
changes in device mix, geographic mix, advertiser spending, ongoing product and policy changes, product mix,
property mix, and changes in foreign currency exchange rates.
Google Subscriptions, Platfor

[page 39]
Total revenues $ 350,018  $ 402,836 
Google Services
Google Advertising
Google Search & other
Google Search & other revenues increased $26.4 billion from 2024 to 2025. The overall growth was driven by

[page 41]
costs, largely for YouTube, depreciation expense, and other technical infrastructure operations costs.
Research and Development
The following table presents research and development expenses (in milli

[page 40]
primarily driven by an increase in subscriptions revenues. This increase was primarily due to the contribution from
growth in paid subscripti

## Create Prompt

For this part, prompt needs to be created so the llm has context to how to answer the question correctly given the context from the retriever.

In [4]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from rag.llm import DEFAULT_CHAT_MODEL, LangChainGenerator
from rag.pipeline import RAGPipeline
from rag.prompts import ANSWER_PROMPT

load_dotenv()

print(ANSWER_PROMPT)

generator = LangChainGenerator(ChatOpenAI(model=DEFAULT_CHAT_MODEL, max_tokens=512))
pipeline = RAGPipeline(retriever, generator, top_k=4)

for i, q in enumerate([
    "how much did Google spend on research and development in 2025?",
    "how much did Google spend on research and development in 2024?",
    "what were Google's main operating costs?",
    "what are some pending acquisitions the company faces?",
]):
    answer = pipeline.answer(q)
    print(f"\n━━━ {i + 1} Q: {q}\n> {answer.text}")

Answer the query using only the context from SEC 10-K filings below. Each passage is prefixed with its source and page; cite the ones you used. Say so if the context does not answer the question. Do not make anything up.

Context:
{context}

Query: {question}

Answer:

━━━ 1 Q: how much did Google spend on research and development in 2025?
> Google spent $61,087 million on research and development in 2025. (Source: [/home/nick/github-projects/Sec-Rag/data/google_10K.pdf p41])

━━━ 2 Q: how much did Google spend on research and development in 2024?
> Google spent $49,326 million on research and development in 2024. (Source: [/home/nick/github-projects/Sec-Rag/data/google_10K.pdf p41])

━━━ 3 Q: what were Google's main operating costs?
> Google's main operating costs included employee compensation expenses, costs associated with traffic acquisition costs (TAC), content acquisition costs, and other expenses. Specifically, for Google Services, employee compensation expenses amounted to $46